In [1]:
import os

In [2]:
os.chdir("../")

In [3]:
pwd

'/Users/aashuanand/AI-Powered-Precision-Agriculture-Monitoring-System'

In [9]:
import ee
import geemap
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
from config.config import CONFIG, FIROZPUR_PLOT, FETCH_DAYS, CLOUD_COVER

import ee
import pandas as pd
from datetime import datetime, timedelta
import os

# ── AUTH ────────────────────────────────────────────────
os.environ["EARTHENGINE_CREDENTIALS"] = "/Users/aashuanand/earthengine_token.json"

try:
    ee.Initialize(project='agrizsquad')
except:
    ee.Authenticate(auth_mode='localhost')
    ee.Initialize(project='agrizsquad')

# ── DATE RANGE (50 DAYS) ────────────────────────────────
end_date = datetime.utcnow()
start_date = end_date - timedelta(days=50)

start_str = start_date.strftime('%Y-%m-%d')
end_str   = end_date.strftime('%Y-%m-%d')

# ── AOI (FIROZPUR) ──────────────────────────────────────
FIROZPUR_PLOT = {
    "center_lat": 30.933,
    "center_lon": 74.622,
    "buffer_m": 5000
}

center = ee.Geometry.Point([
    FIROZPUR_PLOT['center_lon'],
    FIROZPUR_PLOT['center_lat']
])

aoi = center.buffer(FIROZPUR_PLOT['buffer_m'])

# ── SENTINEL-2 ──────────────────────────────────────────
s2 = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
        .filterDate(start_str, end_str)
        .filterBounds(aoi)
        .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20))
        .select(['B2', 'B3', 'B4', 'B8', 'B11', 'B12']))

# ── ADD INDICES + DATE ──────────────────────────────────
def add_indices(image):
    ndvi = image.normalizedDifference(['B8', 'B4']).rename('NDVI')
    ndwi = image.normalizedDifference(['B3', 'B8']).rename('NDWI')

    evi = image.expression(
        '2.5*((NIR-RED)/(NIR+6*RED-7.5*BLUE+1))', {
            'NIR': image.select('B8'),
            'RED': image.select('B4'),
            'BLUE': image.select('B2')
        }).rename('EVI')

    savi = image.expression(
        '1.5*((NIR-RED)/(NIR+RED+0.5))', {
            'NIR': image.select('B8'),
            'RED': image.select('B4')
        }).rename('SAVI')

    date = ee.Date(image.date()).format('YYYY-MM-dd')

    return image.addBands([ndvi, ndwi, evi, savi]).set('date', date)

s2 = s2.map(add_indices)

# ── 20 RANDOM POINTS ────────────────────────────────────
points = ee.FeatureCollection.randomPoints(
    region=aoi,
    points=20,
    seed=42
)

# ── SAMPLE EACH IMAGE ───────────────────────────────────
def sample_image(image):
    sampled = image.sampleRegions(
        collection=points,
        scale=100,
        geometries=True   # ✅ needed for .geo
    )
    return sampled.map(lambda f: f.set('date', image.get('date')))

samples = s2.map(sample_image).flatten()

# ── ADD LAT/LON ─────────────────────────────────────────
def add_lat_lon(feature):
    coords = feature.geometry().coordinates()
    return feature.set({
        'lon': coords.get(0),
        'lat': coords.get(1)
    })

samples = samples.map(add_lat_lon)

# ── EXPORT CSV ──────────────────────────────────────────
task = ee.batch.Export.table.toDrive(
    collection=samples,
    description='Firozpur_full_features',
    folder='AgriZsquad',
    fileNamePrefix='S2_full_dataset',
    fileFormat='CSV'
)

task.start()


import geemap
import os

# ── 9. Convert samples to pandas dataframe ───────────────

df = geemap.ee_to_df(samples)

# ── 10. Save to local data folder ────────────────────────
os.makedirs("data", exist_ok=True)

file_path = "data/sentinel2_data.csv"

df.to_csv(file_path, index=False)

print(f"✅ Data saved locally at: {file_path}")



# # ── 10. Download using gdown ─────────────────────────────
# # 👉 yahan tumhe Google Drive file ID dalna padega manually
# file_id = "YOUR_FILE_ID_HERE"
# url = f"https://drive.google.com/uc?id={file_id}"

# output = "data.csv"
# gdown.download(url, output, quiet=False)

# # ── 11. Load into DataFrame ──────────────────────────────
# df = pd.read_csv(output)

# print(df.head())

# # return df

✅ Data saved locally at: data/sentinel2_data.csv


In [10]:
df.head()

,B11,B12,B2,B3,B4,B8,EVI,NDVI,NDWI,SAVI,date,lat,lon
0,2282,1411,268,687,550,3221,1.479942,0.708300,-0.648414,1.062309,2026-03-07,30.909029,74.596157
1,1968,1257,185,538,442,2828,1.457188,0.729664,-0.680333,1.094328,2026-03-07,30.929880,74.628518
2,2403,1774,557,852,921,2558,1.047345,0.470538,-0.500293,0.705705,2026-03-07,30.947805,74.588666
3,2970,1977,526,949,984,3103,1.046316,0.518473,-0.531589,0.777615,2026-03-07,30.904553,74.607687
4,1397,737,220,512,372,3592,1.928144,0.812311,-0.750487,1.218313,2026-03-07,30.936930,74.574057


In [11]:
df[['lat','lon']].drop_duplicates()

,lat,lon
0,30.909029,74.596157
1,30.929880,74.628518
2,30.947805,74.588666
3,30.904553,74.607687
4,30.936930,74.574057
5,30.956114,74.652497
6,30.927151,74.621201
7,30.929891,74.632704
8,30.934310,74.601285
9,30.929876,74.627471
